# 05 - E/I block fractures and inhibitory structure

**Status: scaffold.** Sections and helper calls are sketched; the analysis is
not written.

Coarse structural fractures: remove or scale each E/I block and the
self-connections, and ask what holds the network's stability together. Port of
the block-removal and Schur-complement sections of `inhib_modulation`.

Blocks are named source-to-target throughout (`EI` = E onto I = rows I, columns
E of a `[post, pre]` matrix). `jc.ablate_block` spells the indexing out so the
naming collision documented in `docs/orientation_convention_proposal.md` cannot
propagate.

The continuous-time framing sharpens the ISN question. In an
inhibition-stabilized network the excitatory subnetwork is unstable on its own
and stability comes from feedback inhibition; here that is testable directly by
comparing $\max\operatorname{Re}\lambda$ of $J_{EE}$ against that of the full $J$.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import jacobian_core as jc

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (7.5, 4.5), "axes.grid": True, "grid.alpha": 0.25})

MATRIX = "matrices/mij_matrix.csv"
NETLIST = "matrices/mij_netlist.csv"

# Synaptic gain. rho(W) < 1 guarantees a stable Jacobian, since eigenvalues of
# J = -I + W are those of W shifted left by one. 0.95 matches the normalization
# used in schur_decomp and inhib_modulation so results stay comparable.
GAIN = 0.95
TAU = 1.0        # membrane time constant; time is measured in units of tau
LEAK = 1.0       # coefficient on -I; leave at 1 unless testing leak sensitivity

data = jc.load_jacobian_data(MATRIX, NETLIST)
labels = data.labels
masks = jc.ei_masks(data.ei, labels)
W, norm_info = jc.normalize_weights(data.W_raw, method="spectral_radius", target=GAIN)
J = jc.build_jacobian(W, tau=TAU, leak=LEAK)
baseline = jc.stability_summary(J)
print(f"alpha = {baseline['spectral_abscissa']:.4f}   omega = {baseline['numerical_abscissa']:.4f}")

OUT = jc.output_dir("05_jacobian_ei_block_fractures")

## 1. Block removals

`EE`, `EI`, `IE`, `II`, self-connections, and the two composite cuts (all
inhibitory output, all inhibitory input).

In [ ]:
# TODO:
# conditions = {b: jc.ablate_block(W, masks, b) for b in ("EE", "EI", "IE", "II")}
# conditions["self"] = jc.ablate_self_connections(W)
# table = jc.ablation_sweep(W, conditions, reference="intact")

## 2. Graded inhibitory gain

Scale `IE` and `II` by a factor sweep. Track where the spectral abscissa crosses
zero: that crossing is the inhibitory gain at which the network loses stability,
and it is the cleanest single number this project can report about the role of
inhibition.

In [ ]:
# TODO: factor sweep on inhibitory blocks; locate the zero crossing of the spectral abscissa.

## 3. ISN test

Compare the excitatory submatrix in isolation against the full network. An
unstable $J_{EE}$ with a stable $J$ is the definition of an
inhibition-stabilized network.

In [ ]:
# TODO: eigenvalues of J[E, E] vs J; report both abscissas.

## 4. Schur complement of inhibition

Eliminate the inhibitory variables to get the effective excitatory Jacobian
$J_{EE} - J_{EI} J_{II}^{-1} J_{IE}$, the continuous-time version of the
inhibitory Schur complement in `inhib_modulation`. Compare its non-normality
against the raw $J_{EE}$: effective inhibitory feedback can create amplification
that neither block shows alone.

In [ ]:
# TODO: partition J by masks; form the Schur complement; run stability and non-normality on it.

## 5. Save and verify

In [ ]:
# TODO: save; assert the block partition recovers J exactly when reassembled.